In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Jefferson Township – Monthly Panel Builder (EMS/Fire + NH + MF + Apt runs)
# Inputs (clean):
#   - nh_data_clean.csv
#   - fire_and_ems_runs_clean.csv  (must include 'ems' dummy: 1=EMS, 0=non‑EMS)
#   - parcels_jefferson_monthly_full.csv
#
# Output:
#   - panel_monthly_with_parcels.csv
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd, numpy as np
from pathlib import Path

# =========================== 0) Config / Paths ================================
ROOT       = Path().resolve().parents[0]
CLEAN_DIR  = ROOT / "data" / "clean"

NH_PATH      = CLEAN_DIR / "nh_data_clean.csv"
RUNS_PATH    = CLEAN_DIR / "fire_and_ems_runs_clean.csv"  # includes 'ems' dummy
PARCELS_PATH = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"
OUT_PATH     = CLEAN_DIR / "panel_monthly_with_parcels.csv"

# Behavior flags
IMPUTE_INDUSTRIAL_AREA     = True   # impute industrial building area if 0/missing
IMPUTE_MIN_SHARE           = 10     # rows needed to build sqft-per-$ imputer

# Beds vs. runs weighting (beds reflect actual values; runs can be coverage-weighted)
TAYLOR_BEDS_WEIGHT = 1.00
TAYLOR_RUNS_WEIGHT = 0.20

# Densities (people/jobs per 1,000 sqft)
DENSITY_RES_PEOPLE_PER_1K_SQFT = 0.9
DENSITY_MF_PEOPLE_PER_1K_SQFT  = 1.4
DENSITY_COM_JOBS_PER_1K_SQFT   = 2.0
DENSITY_IND_JOBS_PER_1K_SQFT   = 1.0

# Codes / cues
LANDUSE_MF_CODE = 429           # Multifamily dwelling (parcels)
RUNS_APT_CODES  = {429}         # Multifamily dwelling (runs)
MF_CUES  = ["APART", "APT", "MULTI", "DUPLEX", "TRIPLEX", "TOWNHOME", "CONDO"]

# ============================ 1) Helpers =====================================
def monthify(dt_series):
    s = pd.to_datetime(dt_series, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def enforce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def any_contains_val(row, cols, cues):
    for c in cols:
        if c in row and pd.notna(row[c]) and any(k in str(row[c]).upper() for k in cues):
            return True
    return False

# ============================ 2) Nursing homes ================================
nh = pd.read_csv(NH_PATH, low_memory=False)
need_min = {"report_month","provider_name"}
missing = need_min - set(nh.columns)
if missing:
    raise ValueError(f"NH input missing columns: {missing}")

nh["provider_name"] = nh["provider_name"].astype(str).str.upper()
nh["report_month"]  = monthify(nh["report_month"])

# Pick best bed column
BED_CANDIDATES = [
    "number_of_certified_beds","total_number_of_beds","licensed_beds",
    "certified_beds","beds","number_of_beds"
]
bed_counts = {c: nh[c].notna().sum() for c in BED_CANDIDATES if c in nh.columns}
if not bed_counts:
    raise ValueError("No recognizable bed column found in nh_data_clean.csv.")
bed_col = max(bed_counts, key=bed_counts.get)
nh[bed_col] = pd.to_numeric(nh[bed_col], errors="coerce")
nh = nh.dropna(subset=["report_month", bed_col])

# Per-facility-month
fac_m = (nh.groupby(["provider_name","report_month"], as_index=False)
           .agg(beds=(bed_col,"max")))

FAC_SAGE   = "SAGE PARK"
FAC_TAYLOR = "TAYLOR SPRINGS"

fac_m["beds_sage_park_raw"] = np.where(fac_m["provider_name"].str.contains(FAC_SAGE),   fac_m["beds"], 0)
fac_m["beds_taylor_raw"]    = np.where(fac_m["provider_name"].str.contains(FAC_TAYLOR), fac_m["beds"], 0)

nh_monthly = (fac_m.groupby("report_month", as_index=False)
                .agg(beds_sage_park_raw=("beds_sage_park_raw","sum"),
                     beds_taylor_raw=("beds_taylor_raw","sum")))

# Beds: unweighted
nh_monthly["beds_sage_park"]       = nh_monthly["beds_sage_park_raw"]
nh_monthly["beds_taylor_springs"]  = nh_monthly["beds_taylor_raw"] * TAYLOR_BEDS_WEIGHT
nh_monthly["nh_total_certified_beds"] = nh_monthly["beds_sage_park"] + nh_monthly["beds_taylor_springs"]

# Fill continuous monthly index and ffill
if not nh_monthly.empty:
    full = pd.date_range(nh_monthly["report_month"].min(),
                         nh_monthly["report_month"].max(), freq="MS")
    nh_monthly = (nh_monthly.set_index("report_month")
                              .reindex(full)
                              .rename_axis("month")
                              .reset_index())
    cols_ff = ["beds_sage_park","beds_taylor_raw","beds_taylor_springs","nh_total_certified_beds"]
    nh_monthly[cols_ff] = nh_monthly[cols_ff].ffill()
else:
    nh_monthly = pd.DataFrame(columns=["month","beds_sage_park","beds_taylor_raw","beds_taylor_springs","nh_total_certified_beds"])

# ============================ 3) Runs & EMS/Fire ==============================
runs = pd.read_csv(RUNS_PATH, low_memory=False)
need = {"incident_date","incident_number","property_use_code","address","ems"}
missing = need - set(runs.columns)
if missing:
    raise ValueError(f"Runs input missing columns: {missing}")

runs["month"] = monthify(runs["incident_date"])
runs = runs.dropna(subset=["month"])
runs["ems"] = pd.to_numeric(runs["ems"], errors="coerce").fillna(0).astype(int)

# Address normalization (vectorized)
addr = runs["address"].astype(str).str.upper().str.strip()
addr = addr.str.replace(r"\s+", " ", regex=True)
for old, new in [
    (" AVENUE", " AVE"), (" AVE.", " AVE"),
    (" ROAD", " RD"),    (" RD.", " RD"),
    (" STREET", " ST"),  (" ST.", " ST"),
    (" DRIVE", " DR"),   (" DR.", " DR"),
    (" LANE", " LN"),    (" LN.", " LN"),
]:
    addr = addr.str.replace(old, new, regex=False)
runs["address_norm"] = addr

# Facility prefix matches
sage_regex   = r"^\s*5201\s+(?:E\s+)?MORSE\b"  # allow '5201 E MORSE'
taylor_regex = r"^\s*748\s+TAYLOR\b"
runs["to_sage_park"]  = runs["address_norm"].str.contains(sage_regex, regex=True, na=False).astype(int)
runs["to_taylor_raw"] = runs["address_norm"].str.contains(taylor_regex, regex=True, na=False).astype(int)

# Row-wise EMS/Fire + facility splits
runs["ems_call"]  = runs["ems"]
runs["fire_call"] = 1 - runs["ems_call"]

runs["sage_total"]   = runs["to_sage_park"]
runs["taylor_total"] = runs["to_taylor_raw"]
runs["sage_ems"]     = runs["sage_total"]   * runs["ems_call"]
runs["sage_fire"]    = runs["sage_total"]   * runs["fire_call"]
runs["taylor_ems"]   = runs["taylor_total"] * runs["ems_call"]
runs["taylor_fire"]  = runs["taylor_total"] * runs["fire_call"]

# Monthly totals (pure reductions)
monthly_total = (runs.groupby("month", as_index=False)
                      .agg(total_calls=("incident_number","count"),
                           ems_calls=("ems_call","sum"),
                           fire_calls=("fire_call","sum")))

# Nursing-home runs (raw + weighted Taylor)
nh_runs = (runs.groupby("month", as_index=False)
              .agg(
                  runs_sage_total       = ("sage_total","sum"),
                  runs_taylor_total_raw = ("taylor_total","sum"),
                  runs_sage_ems         = ("sage_ems","sum"),
                  runs_taylor_ems_raw   = ("taylor_ems","sum"),
              ))
nh_runs["runs_sage_fire"]       = nh_runs["runs_sage_total"]      - nh_runs["runs_sage_ems"]
nh_runs["runs_taylor_fire_raw"] = nh_runs["runs_taylor_total_raw"] - nh_runs["runs_taylor_ems_raw"]
nh_runs["runs_taylor_total"]    = (nh_runs["runs_taylor_total_raw"] * TAYLOR_RUNS_WEIGHT).round(0).astype(int)
nh_runs["runs_taylor_ems"]      = (nh_runs["runs_taylor_ems_raw"]   * TAYLOR_RUNS_WEIGHT).round(0).astype(int)
nh_runs["runs_taylor_fire"]     = nh_runs["runs_taylor_total"] - nh_runs["runs_taylor_ems"]

# Apartment runs — EXACT code match (429)
runs["property_use_code"] = pd.to_numeric(runs["property_use_code"], errors="coerce")
APT_CODE = 429  # Multifamily dwelling

runs["to_apartment"] = runs["property_use_code"].eq(APT_CODE).astype(int)
runs["apt_ems"]      = runs["to_apartment"] * runs["ems_call"]
runs["apt_fire"]     = runs["to_apartment"] * runs["fire_call"]

apt_runs = (runs.groupby("month", as_index=False)
                .agg(
                    runs_apartment_total=("to_apartment", "sum"),
                    runs_apartment_ems=("apt_ems", "sum"),
                    runs_apartment_fire=("apt_fire", "sum"),
                ))

# ============================ 4) Parcels & land use ===========================
parcels = pd.read_csv(PARCELS_PATH, low_memory=False)
date_col = "snapshot_month" if "snapshot_month" in parcels.columns else "report_month"
parcels["month"] = monthify(parcels[date_col])

num_cols = ["apprlnd","apprbld","apprtot","area_a","acrea","land_sqft","landuse"]
parcels = enforce_numeric(parcels, num_cols)

# land sqft convenience
if "land_sqft" in parcels.columns:
    parcels["land_sqft_use"] = parcels["land_sqft"]
elif "acrea" in parcels.columns:
    parcels["land_sqft_use"] = parcels["acrea"] * 43560
else:
    parcels["land_sqft_use"] = np.nan

# Base category from pclass
PCLASS_MAP = {
    "R": "1 Residential",
    "A": "2 Agricultural",
    "M": "3 Mineral",
    "C": "4 Commercial",
    "I": "5 Industrial",
    "U": "6 Public Utility Real",
    "P": "7 Public Utility Personal",
    "E": "8 General Personal",
    "Z": "8 General Personal"
}
parcels["pclass"] = parcels["pclass"].astype(str).str.upper()
parcels["auditor_category"] = parcels["pclass"].map(PCLASS_MAP).fillna("8 General Personal")

# Impute industrial building area (area_a_used)
if IMPUTE_INDUSTRIAL_AREA:
    src  = parcels.copy()
    comp = src[src["auditor_category"].isin(["4 Commercial","1 Residential"])].copy()
    comp = comp[(comp["area_a"].fillna(0) > 0) & (comp["apprbld"].fillna(0) > 0)]
    if len(comp) >= IMPUTE_MIN_SHARE:
        comp["sqft_per_dollar"] = comp["area_a"] / comp["apprbld"]
        med = (comp.groupby("auditor_category", as_index=False)["sqft_per_dollar"]
                  .median().sort_values("sqft_per_dollar", ascending=False))
        if "4 Commercial" in med["auditor_category"].values:
            sqft_per_dollar = float(med.loc[med["auditor_category"]=="4 Commercial","sqft_per_dollar"].iloc[0])
        elif "1 Residential" in med["auditor_category"].values:
            sqft_per_dollar = float(med.loc[med["auditor_category"]=="1 Residential","sqft_per_dollar"].iloc[0])
        else:
            sqft_per_dollar = np.nan
    else:
        sqft_per_dollar = np.nan

    parcels["area_a_imputed"] = np.nan
    mask_ind = parcels["auditor_category"].eq("5 Industrial") & (parcels["area_a"].fillna(0) == 0)
    if not np.isnan(sqft_per_dollar):
        can_use_dollar = mask_ind & parcels["apprbld"].notna() & (parcels["apprbld"] > 0)
        parcels.loc[can_use_dollar, "area_a_imputed"] = parcels.loc[can_use_dollar, "apprbld"] * sqft_per_dollar
    fallback = mask_ind & parcels["land_sqft_use"].notna() & parcels["area_a_imputed"].isna()
    parcels.loc[fallback, "area_a_imputed"] = parcels.loc[fallback, "land_sqft_use"]
    parcels["area_a_used"] = np.where(mask_ind & parcels["area_a_imputed"].notna(),
                                      parcels["area_a_imputed"], parcels["area_a"])
else:
    parcels["area_a_used"] = parcels["area_a"]

# ── Multifamily detection (exact code + text backup on your real columns) ────
PARCEL_TEXT_COLS = ["descr1", "descr2", "descr3", "landuse", "proptyp"]

is_mf_code = parcels["landuse"].eq(LANDUSE_MF_CODE)
is_mf_text = parcels.apply(lambda r: any_contains_val(r, PARCEL_TEXT_COLS, MF_CUES), axis=1)

parcels["res_subtype"] = np.where(
    parcels["auditor_category"].eq("1 Residential") & (is_mf_code | is_mf_text),
    "res_mf",
    np.where(parcels["auditor_category"].eq("1 Residential"), "res_sf", "")
)

# Compute sqft by subtype
parcels["res_sf_area_sqft"] = np.where(
    (parcels["auditor_category"].eq("1 Residential")) & (parcels["res_subtype"]=="res_sf"),
    parcels["area_a_used"], 0.0
)
parcels["res_mf_area_sqft"] = np.where(
    (parcels["auditor_category"].eq("1 Residential")) & (parcels["res_subtype"]=="res_mf"),
    parcels["area_a_used"], 0.0
)

# Commercial (remove any MF-like records that slipped into commercial)
parcels["com_area_sqft_raw"] = np.where(parcels["auditor_category"].eq("4 Commercial"),
                                       parcels["area_a_used"], 0.0)
suspect_mf_in_com = is_mf_code | parcels.apply(
    lambda r: any_contains_val(r, PARCEL_TEXT_COLS, MF_CUES), axis=1
)
parcels["com_area_sqft_no_mf"] = np.where(suspect_mf_in_com, 0.0, parcels["com_area_sqft_raw"])

# Industrial
parcels["ind_area_sqft"] = np.where(parcels["auditor_category"].eq("5 Industrial"),
                                    parcels["area_a_used"], 0.0)

# Monthly aggregates
agg_cols = {
    "res_sf_area_sqft":"sum",
    "res_mf_area_sqft":"sum",
    "com_area_sqft_no_mf":"sum",
    "ind_area_sqft":"sum",
    "apprbld":"sum",
    "apprtot":"sum"
}
parcel_m = (parcels.groupby("month", as_index=False)
                   .agg(agg_cols)
                   .rename(columns={"com_area_sqft_no_mf":"com_area_sqft"}))

# ============================ 5) Population / Worker proxies ==================
parcel_m["est_residents_sf"]     = (parcel_m["res_sf_area_sqft"] / 1_000.0) * DENSITY_RES_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_mf"]     = (parcel_m["res_mf_area_sqft"] / 1_000.0) * DENSITY_MF_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_total"]  = parcel_m["est_residents_sf"] + parcel_m["est_residents_mf"]

parcel_m["est_workers_com"]      = (parcel_m["com_area_sqft"] / 1_000.0) * DENSITY_COM_JOBS_PER_1K_SQFT
parcel_m["est_workers_ind"]      = (parcel_m["ind_area_sqft"] / 1_000.0) * DENSITY_IND_JOBS_PER_1K_SQFT
parcel_m["est_workers_total"]    = parcel_m["est_workers_com"] + parcel_m["est_workers_ind"]

# ============================ 6) Merge panel =================================
panel = (monthly_total
         .merge(nh_runs, on="month", how="left")
         .merge(nh_monthly, on="month", how="left")
         .merge(parcel_m, on="month", how="left")
         .merge(apt_runs, on="month", how="left")   # apartment runs by code
         .sort_values("month")
         .reset_index(drop=True))

# Scaled convenience fields (per 1k sqft; per $1M)
for c in ["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]:
    panel[f"{c}_k"] = panel[c] / 1_000.0
for c in ["apprbld","apprtot"]:
    panel[f"{c}_M"] = panel[c] / 1_000_000.0

# Optional lags & YoY deltas
ADD_LAGS = True
ADD_YOY  = True

if ADD_LAGS:
    for c in ["res_sf_area_sqft_k","res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k",
              "apprbld_M","apprtot_M",
              "est_residents_total","est_workers_total",
              "beds_sage_park","beds_taylor_springs","nh_total_certified_beds",
              "ems_calls","fire_calls","total_calls",
              "runs_apartment_total","runs_apartment_ems","runs_apartment_fire"]:
        if c in panel.columns:
            panel[f"{c}_lag6"] = panel[c].shift(6)

if ADD_YOY:
    for c in ["res_sf_area_sqft_k","res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k",
              "apprbld_M","apprtot_M",
              "est_residents_total","est_workers_total",
              "beds_sage_park","beds_taylor_springs","nh_total_certified_beds",
              "ems_calls","fire_calls","total_calls",
              "runs_apartment_total","runs_apartment_ems","runs_apartment_fire"]:
        if c in panel.columns:
            panel[f"{c}_yoy"] = panel[c] - panel[c].shift(12)

# ============================ 7) Save ========================================
panel.to_csv(OUT_PATH, index=False)
print(f"Saved monthly panel to: {OUT_PATH}")
print(f"Rows: {len(panel):,}  |  Columns: {len(panel.columns):,}")
keep_cols = ("^month$|ems_calls$|fire_calls$|runs_.*|beds_.*|res_.*_k$|com_area_sqft_k$|ind_area_sqft_k$|"
             "est_residents_total$|est_workers_total$|runs_apartment_.*")
print(panel.filter(regex=keep_cols).head(8))

Saved monthly panel to: C:\Repositories\jefferson-township-run-forecasting\data\clean\panel_monthly_with_parcels.csv
Rows: 84  |  Columns: 73
       month  ems_calls  fire_calls  runs_sage_total  runs_taylor_total_raw  \
0 2018-08-01        115          91               10                      0   
1 2018-09-01        126          90                6                      0   
2 2018-10-01        123          82                7                      0   
3 2018-11-01        120          85                2                      0   
4 2018-12-01        116          86                5                      0   
5 2019-01-01        100          99                4                      0   
6 2019-02-01         92          76                3                      0   
7 2019-03-01        113          76                3                      0   

   runs_sage_ems  runs_taylor_ems_raw  runs_sage_fire  runs_taylor_fire_raw  \
0             10                    0               0              

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Jefferson Township — Purpose‑Built Monthly Panel (answers your 9 questions)
#
# Inputs (from CLEAN_DIR):
#   - nh_data_clean.csv
#   - fire_and_ems_runs_clean.csv   (must include 'ems' dummy: 1=EMS, 0=non‑EMS)
#   - parcels_jefferson_monthly_full.csv
#
# Output (to CLEAN_DIR):
#   - panel_monthly_with_parcels.csv
#
# Design goals (mapped to your 9 questions):
#   Q1–Q2: Facility runs (Sage Park, Taylor Springs) ~ Beds (per facility)
#   Q3:  Apt EMS ~ MF residents (from MF sqft * density)
#   Q4:  Apt Fire ~ MF sqft
#   Q5:  EMS to Commercial ~ Commercial workers (from COM sqft * jobs density)
#   Q6:  Fire to Commercial ~ Commercial sqft
#   Q7:  EMS to Residential (non‑apt, non‑NH) ~ Total residents (SF+MF)
#   Q8:  Fire to Residential (SF+MF) ~ Residential sqft (SF+MF)
#   Q9:  Fire to Industrial ~ Industrial sqft
#
# Guardrails:
#   • No double‑counting (apartments not in commercial; NH not in residential).
#   • Forward‑fill stable inventories (beds, parcel aggregates) to avoid NA gaps.
#   • Clean, minimal fields: outcomes + matched predictors for each question.
# ─────────────────────────────────────────────────────────────────────────────

import re
import numpy as np
import pandas as pd
from pathlib import Path

# =========================== 0) Paths ===========================
ROOT       = Path().resolve().parents[0]
CLEAN_DIR  = ROOT / "data" / "clean"

NH_PATH      = CLEAN_DIR / "nh_data_clean.csv"
RUNS_PATH    = CLEAN_DIR / "fire_and_ems_runs_clean.csv"   # includes 'ems' dummy
PARCELS_PATH = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"
OUT_PATH     = CLEAN_DIR / "panel_monthly_with_parcels.csv"

# =========================== 1) Config ==========================
# Facilities (address regex for run tagging)
FAC_REGEX = {
    "SAGE PARK": r"^\s*5201\s+(?:E\s+)?MORSE\b",
    "TAYLOR SPRINGS": r"^\s*748\s+TAYLOR\b",
}

# Apartments (runs) — Property Use Code
APT_CODE = 429  # multifamily dwelling (runs)

# Residential landuse bands (parcels) — Franklin Co. common
SF_BAND = (510, 515)  # single‑family
MF_BAND = (550, 553)  # multifamily

# Density proxies (per 1,000 sqft of building area)
DENSITY_RES_PEOPLE_PER_1K_SQFT = 0.9
DENSITY_MF_PEOPLE_PER_1K_SQFT  = 1.4
DENSITY_COM_JOBS_PER_1K_SQFT   = 2.0
DENSITY_IND_JOBS_PER_1K_SQFT   = 1.0

# Industrial area imputation (if area=0 but $apprbld>0)
IMPUTE_INDUSTRIAL_AREA = True

# ======================== 2) Helpers ============================
def monthify(s):
    s = pd.to_datetime(s, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def enforce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def normalize_address(s):
    s = s.astype(str).str.upper().str.strip().str.replace(r"\s+", " ", regex=True)
    rep = [(" AVENUE"," AVE"),(" AVE."," AVE"),
           (" STREET"," ST"),(" ST."," ST"),
           (" ROAD"," RD"),(" RD."," RD"),
           (" DRIVE"," DR"),(" DR."," DR"),
           (" LANE"," LN"),(" LN."," LN")]
    for o,n in rep:
        s = s.str.replace(o, n, regex=False)
    return s

def tag_facility(addr):
    if not isinstance(addr, str): return None
    for fac, pat in FAC_REGEX.items():
        if re.search(pat, addr, re.IGNORECASE): return fac
    return None

# ===================== 3) Load inputs ===========================
runs    = pd.read_csv(RUNS_PATH, low_memory=False)
nh      = pd.read_csv(NH_PATH, low_memory=False)
parcels = pd.read_csv(PARCELS_PATH, low_memory=False)

# ===================== 4) Nursing homes (beds, by facility) ==================
nh["provider_name"] = nh["provider_name"].astype(str).str.upper().str.strip()
date_col = "report_month" if "report_month" in nh.columns else "processing_date"
nh["month"] = monthify(nh[date_col])

BED_CANDIDATES = [
    "number_of_certified_beds","total_number_of_beds","licensed_beds",
    "certified_beds","beds","number_of_beds"
]
bed_counts = {c: nh[c].notna().sum() for c in BED_CANDIDATES if c in nh.columns}
if not bed_counts:
    raise ValueError("No recognizable bed column in NH file.")
bed_col = max(bed_counts, key=bed_counts.get)
nh[bed_col] = pd.to_numeric(nh[bed_col], errors="coerce")

nh_fac_m = (nh.dropna(subset=["month"])
              .groupby(["provider_name","month"], as_index=False)
              .agg(beds=(bed_col,"max")))

def canonical_fac(nm):
    nm = str(nm).upper()
    if re.search(r"SAGE\s*PARK", nm): return "SAGE PARK"
    if re.search(r"TAYLOR\s*SPRINGS", nm): return "TAYLOR SPRINGS"
    return nm
nh_fac_m["facility"] = nh_fac_m["provider_name"].apply(canonical_fac)

# Collapse to monthly beds for the two targeted facilities, then forward‑fill
beds_m = (nh_fac_m.pivot_table(index="month", columns="facility", values="beds", aggfunc="max")
                 .rename(columns={"SAGE PARK":"beds_sage_park", "TAYLOR SPRINGS":"beds_taylor_springs"})
                 .sort_index())
if not beds_m.empty:
    # Forward‑fill within observed window
    beds_m = beds_m.ffill()
beds_m = beds_m.reset_index()

# ===================== 5) Runs: EMS/Fire, facility & type tags ===============
runs["month"] = monthify(runs["incident_date"])
runs = runs.dropna(subset=["month"]).copy()

# EMS/Fire flags
runs["ems"] = pd.to_numeric(runs.get("ems", 0), errors="coerce").fillna(0).astype(int)
runs["is_ems"]  = runs["ems"]
runs["is_fire"] = 1 - runs["is_ems"]

# Address → facility tags (Sage / Taylor)
runs["address_norm"] = normalize_address(runs["address"])
runs["facility"] = runs["address_norm"].apply(tag_facility)

fac_runs = (runs.dropna(subset=["facility"])
                 .groupby(["month","facility"], as_index=False)
                 .agg(
                     runs_fac_total=("incident_number","count"),
                     runs_fac_ems=("is_ems","sum"),
                     runs_fac_fire=("is_fire","sum"),
                 ))

# Apartment runs via PUC
runs["property_use_code"] = pd.to_numeric(runs.get("property_use_code", np.nan), errors="coerce")
runs["is_apartment"] = runs["property_use_code"].eq(APT_CODE).astype(int)

apt_runs = (runs.groupby("month", as_index=False)
                .agg(
                    runs_apartment_total=("is_apartment","sum"),
                    runs_apartment_ems=("is_apartment", lambda s: int((s * runs.loc[s.index, "is_ems"]).sum())),
                    runs_apartment_fire=("is_apartment", lambda s: int((s * runs.loc[s.index, "is_fire"]).sum())),
                ))

# Township totals
tot_runs = (runs.groupby("month", as_index=False)
                 .agg(
                     total_calls=("incident_number","count"),
                     ems_calls=("is_ems","sum"),
                     fire_calls=("is_fire","sum"),
                 ))

# Facility‑specific monthly runs (wide)
fac_wide = (fac_runs.pivot_table(index="month", columns="facility",
                                 values=["runs_fac_total","runs_fac_ems","runs_fac_fire"],
                                 aggfunc="sum").sort_index())

# Flatten MultIndex columns
if not fac_wide.empty:
    fac_wide.columns = [f"{lvl0}_{lvl1}".lower().replace(" ","_")
                        for (lvl0, lvl1) in fac_wide.columns.to_flat_index()]
    fac_wide = fac_wide.reset_index()

# Keep Sage and Taylor columns explicitly (NA→0)
for base in ["runs_fac_total","runs_fac_ems","runs_fac_fire"]:
    for fac in ["sage_park","taylor_springs"]:
        col = f"{base}_{fac}"
        if col not in fac_wide.columns:
            fac_wide[col] = 0

# ===================== 6) Parcels → sqft & population/workers =================
parcels["month"] = monthify(parcels["snapshot_month"] if "snapshot_month" in parcels.columns
                            else parcels["report_month"])

num_cols = ["apprlnd","apprbld","apprtot","area_a","acrea","land_sqft","landuse","pclass"]
parcels = enforce_numeric(parcels, num_cols)

# land sqft convenience
if "land_sqft" in parcels.columns:
    parcels["land_sqft_use"] = parcels["land_sqft"]
elif "acrea" in parcels.columns:
    parcels["land_sqft_use"] = parcels["acrea"] * 43560
else:
    parcels["land_sqft_use"] = np.nan

# pclass mapping
PCLASS_MAP = {
    "R":"Residential","A":"Agricultural","M":"Mineral",
    "C":"Commercial", "I":"Industrial","U":"Utility Real",
    "P":"Utility Personal","E":"General Personal","Z":"General Personal"
}
parcels["pclass_txt"] = parcels["pclass"].astype(str).str.upper().map(PCLASS_MAP).fillna("General Personal")
lu = parcels["landuse"]

# Industrial area imputation if missing/zero
parcels["area_a_used"] = parcels["area_a"].copy()
if IMPUTE_INDUSTRIAL_AREA:
    comp = parcels[(parcels["area_a"].fillna(0)>0) & (parcels["apprbld"].fillna(0)>0)
                   & parcels["pclass_txt"].isin(["Commercial","Residential"])].copy()
    if len(comp) > 50:
        comp["sqft_per_dollar"] = comp["area_a"] / comp["apprbld"]
        med = comp.groupby("pclass_txt")["sqft_per_dollar"].median()
        sqft_per_dollar = med.get("Commercial", med.get("Residential", np.nan))
    else:
        sqft_per_dollar = np.nan

    mask_ind0 = (parcels["pclass_txt"]=="Industrial") & (parcels["area_a_used"].fillna(0)==0)
    if not np.isnan(sqft_per_dollar):
        can = mask_ind0 & parcels["apprbld"].notna() & (parcels["apprbld"]>0)
        parcels.loc[can, "area_a_used"] = parcels.loc[can, "apprbld"] * sqft_per_dollar
    land_fb = mask_ind0 & parcels["area_a_used"].isna() & parcels["land_sqft_use"].notna()
    parcels.loc[land_fb, "area_a_used"] = parcels.loc[land_fb, "land_sqft_use"]

# SF / MF by landuse bands (within Residential)
is_sf = lu.between(SF_BAND[0], SF_BAND[1], inclusive="both")
is_mf = lu.between(MF_BAND[0], MF_BAND[1], inclusive="both")

parcels["res_sf_area_sqft"] = np.where((parcels["pclass_txt"]=="Residential") & is_sf, parcels["area_a_used"], 0.0)
parcels["res_mf_area_sqft"] = np.where((parcels["pclass_txt"]=="Residential") & is_mf, parcels["area_a_used"], 0.0)
parcels["com_area_sqft"]    = np.where(parcels["pclass_txt"]=="Commercial",  parcels["area_a_used"], 0.0)
parcels["ind_area_sqft"]    = np.where(parcels["pclass_txt"]=="Industrial", parcels["area_a_used"], 0.0)

# Monthly parcel aggregates
parcel_m = (parcels.groupby("month", as_index=False)
                   .agg(res_sf_area_sqft=("res_sf_area_sqft","sum"),
                        res_mf_area_sqft=("res_mf_area_sqft","sum"),
                        com_area_sqft=("com_area_sqft","sum"),
                        ind_area_sqft=("ind_area_sqft","sum"))
            .sort_values("month"))

# Residents / workers proxies (per 1,000 sqft)
parcel_m["est_residents_sf"]     = (parcel_m["res_sf_area_sqft"]/1000.0) * DENSITY_RES_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_mf"]     = (parcel_m["res_mf_area_sqft"]/1000.0) * DENSITY_MF_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_total"]  = parcel_m["est_residents_sf"] + parcel_m["est_residents_mf"]
parcel_m["est_workers_com"]      = (parcel_m["com_area_sqft"]/1000.0) * DENSITY_COM_JOBS_PER_1K_SQFT
parcel_m["est_workers_ind"]      = (parcel_m["ind_area_sqft"]/1000.0) * DENSITY_IND_JOBS_PER_1K_SQFT

# k‑sqft convenience
for c in ["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]:
    parcel_m[f"{c}_k"] = parcel_m[c] / 1000.0
parcel_m["total_res_area_sqft_k"] = parcel_m["res_sf_area_sqft_k"] + parcel_m["res_mf_area_sqft_k"]

# ===================== 7) Compose analysis panel =============================
# Base: township totals
panel = tot_runs.merge(apt_runs, on="month", how="left").merge(fac_wide, on="month", how="left").merge(beds_m, on="month", how="left")
panel = panel.merge(parcel_m, on="month", how="left").sort_values("month").reset_index(drop=True)

# Fill NA zeros where appropriate
zero_cols_int = [
    "runs_apartment_total","runs_apartment_ems","runs_apartment_fire",
    "runs_fac_total_sage_park","runs_fac_ems_sage_park","runs_fac_fire_sage_park",
    "runs_fac_total_taylor_springs","runs_fac_ems_taylor_springs","runs_fac_fire_taylor_springs",
]
for c in zero_cols_int:
    if c not in panel.columns: panel[c] = 0
panel[zero_cols_int] = panel[zero_cols_int].fillna(0).astype(int)

# Forward‑fill beds across months
for c in ["beds_sage_park","beds_taylor_springs"]:
    if c in panel.columns:
        panel[c] = panel[c].ffill()

# Derived NH totals (if you ever need)
panel["runs_sage_total"]   = panel["runs_fac_total_sage_park"]
panel["runs_taylor_total"] = panel["runs_fac_total_taylor_springs"]

# Residential outcomes excluding NH & Apartments (no double‑count)
# EMS: total EMS − apartment EMS − NH EMS (Sage + Taylor)
panel["runs_sage_ems"]   = panel["runs_fac_ems_sage_park"]
panel["runs_taylor_ems"] = panel["runs_fac_ems_taylor_springs"]
panel["ems_residential_other"] = (
    panel["ems_calls"].fillna(0)
    - panel["runs_apartment_ems"].fillna(0)
    - panel["runs_sage_ems"].fillna(0)
    - panel["runs_taylor_ems"].fillna(0)
)

# FIRE (residential other): total fire − apartment fire − NH fire
panel["runs_sage_fire"]   = panel["runs_fac_fire_sage_park"]
panel["runs_taylor_fire"] = panel["runs_fac_fire_taylor_springs"]
panel["fire_residential_other"] = (
    panel["fire_calls"].fillna(0)
    - panel["runs_apartment_fire"].fillna(0)
    - panel["runs_sage_fire"].fillna(0)
    - panel["runs_taylor_fire"].fillna(0)
)

# ===================== 8) Columns needed for each question ===================
# Q1/Q2: facility beds + facility total runs
#   beds_sage_park, runs_sage_total
#   beds_taylor_springs, runs_taylor_total

# Q3: apt EMS (outcome) + MF residents (predictor)
#   runs_apartment_ems, est_residents_mf

# Q4: apt Fire (outcome) + MF area_k (predictor)
#   runs_apartment_fire, res_mf_area_sqft_k

# Q5: EMS to Commercial ~ Commercial workers (proxy from parcels)
#   ems_calls_commercial (NOT available in raw runs) → use total EMS as proxy or derive later
#   Here we retain est_workers_com so you can fit EMS_total ~ est_workers_com or create EMS_com later.
#   Columns: ems_calls (outcome candidate), est_workers_com (predictor)

# Q6: Fire to Commercial ~ com_area_sqft_k
#   fire_calls (outcome proxy), com_area_sqft_k (predictor)

# Q7: EMS to Residential (non‑apt, non‑NH) ~ total residents
#   ems_residential_other, est_residents_total

# Q8: Fire to Residential (SF+MF) ~ total residential sqft
#   fire_residential_other, total_res_area_sqft_k

# Q9: Fire to Industrial ~ ind_area_sqft_k
#   fire_calls (outcome proxy), ind_area_sqft_k

# ===================== 9) Save & quick sanity print ==========================
panel.to_csv(OUT_PATH, index=False)

print(f"Saved monthly panel to: {OUT_PATH}")
print(f"Rows: {len(panel):,}  |  Columns: {panel.shape[1]}")

show_cols = [
    "month",
    # Q1/Q2
    "beds_sage_park","runs_sage_total","beds_taylor_springs","runs_taylor_total",
    # Q3/Q4
    "runs_apartment_ems","runs_apartment_fire","est_residents_mf","res_mf_area_sqft_k",
    # Q5/Q6
    "ems_calls","fire_calls","est_workers_com","com_area_sqft_k",
    # Q7/Q8
    "ems_residential_other","fire_residential_other","est_residents_total","total_res_area_sqft_k",
    # Q9
    "ind_area_sqft_k",
]
have = [c for c in show_cols if c in panel.columns]
print(panel[have].head(8))

Saved monthly panel to: C:\Repositories\jefferson-township-run-forecasting\data\clean\panel_monthly_with_parcels.csv
Rows: 84  |  Columns: 37
       month  beds_sage_park  runs_sage_total  beds_taylor_springs  \
0 2018-08-01            50.0               10                  NaN   
1 2018-09-01            50.0                6                  NaN   
2 2018-10-01            50.0                7                  NaN   
3 2018-11-01            50.0                2                  NaN   
4 2018-12-01            50.0                5                  NaN   
5 2019-01-01            50.0                4                  NaN   
6 2019-02-01            50.0                3                  NaN   
7 2019-03-01            50.0                3                  NaN   

   runs_taylor_total  runs_apartment_ems  runs_apartment_fire  \
0                  0                   3                    3   
1                  0                   8                    5   
2                  0            

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Jefferson Township — Panel Builder (updated for 9-question models)
# Inputs: nh_data_clean.csv, fire_and_ems_runs_clean.csv, parcels_jefferson_monthly_full.csv
# Output: panel_monthly_with_parcels.csv
# ─────────────────────────────────────────────────────────────────────────────

import re
import numpy as np
import pandas as pd
from pathlib import Path

# =========================== Paths ===========================
ROOT       = Path().resolve().parents[0]
CLEAN_DIR  = ROOT / "data" / "clean"

NH_PATH      = CLEAN_DIR / "nh_data_clean.csv"
RUNS_PATH    = CLEAN_DIR / "fire_and_ems_runs_clean.csv"   # includes 'ems' dummy
PARCELS_PATH = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"
OUT_PATH     = CLEAN_DIR / "panel_monthly_with_parcels.csv"

# =========================== Config ==========================
FAC_REGEX = {
    "SAGE PARK": r"^\s*5201\s+(?:E\s+)?MORSE\b",
    "TAYLOR SPRINGS": r"^\s*748\s+TAYLOR\b"
}
APT_CODE = 429

# Appraiser landuse bands (Auditor)
SF_BAND = (510, 515)
MF_BAND = (550, 553)

# People/jobs per 1,000 sqft
DENSITY_RES_PEOPLE_PER_1K_SQFT = 0.9
DENSITY_MF_PEOPLE_PER_1K_SQFT  = 1.4
DENSITY_COM_JOBS_PER_1K_SQFT   = 2.0
DENSITY_IND_JOBS_PER_1K_SQFT   = 1.0

IMPUTE_INDUSTRIAL_AREA = True

# Buckets from runs “property_use_category”
COMMERCIAL_CATS = {
    "5 - MERCANTILE, BUSINESS",
    "1 - ASSEMBLY",
    "2 - EDUCATIONAL",
    "8 - STORAGE",
    "9 - OUTSIDE OR SPECIAL PROPERTY",
    "3 - HEALTH CARE, DETENTION & CORRECTION",
}
INDUSTRIAL_CATS = {
    "7 - MANUFACTURING, PROCESSING",
    "6 - INDUSTRIAL, UTILITY, DEFENSE, AGRICULTURE, MINING",
}
RESIDENTIAL_PREFIX = "4 - RESIDENTIAL"

# =========================== Helpers =========================
def monthify(s):
    s = pd.to_datetime(s, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def enforce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def normalize_address(s):
    s = s.astype(str).str.upper().str.strip().str.replace(r"\s+", " ", regex=True)
    rep = [(" AVENUE"," AVE"),(" AVE."," AVE"),
           (" STREET"," ST"),(" ST."," ST"),
           (" ROAD"," RD"),(" RD."," RD"),
           (" DRIVE"," DR"),(" DR."," DR"),
           (" LANE"," LN"),(" LN."," LN")]
    for o,n in rep: s = s.str.replace(o, n, regex=False)
    return s

def tag_facility(addr):
    if not isinstance(addr, str): return None
    for fac, pat in FAC_REGEX.items():
        if re.search(pat, addr, re.IGNORECASE): return fac
    return None

def _first_existing(df, cols):
    for c in cols:
        if c in df.columns:
            return c
    return None

# ============================ Load ============================
runs    = pd.read_csv(RUNS_PATH, low_memory=False)
nh      = pd.read_csv(NH_PATH, low_memory=False)
parcels_raw = pd.read_csv(PARCELS_PATH, low_memory=False)

# ======================= Nursing homes =======================
nh["provider_name"] = nh["provider_name"].astype(str).str.upper().str.strip()
nh_date_col = "report_month" if "report_month" in nh.columns else _first_existing(nh, ["processing_date","month","date"])
nh["month"] = monthify(nh[nh_date_col])

BED_CANDIDATES = ["number_of_certified_beds","total_number_of_beds","licensed_beds",
                  "certified_beds","beds","number_of_beds"]
bed_counts = {c: nh[c].notna().sum() for c in BED_CANDIDATES if c in nh.columns}
if not bed_counts:
    raise ValueError("No recognizable bed column in NH file.")
bed_col = max(bed_counts, key=bed_counts.get)
nh[bed_col] = pd.to_numeric(nh[bed_col], errors="coerce")

nh_fac_m = (nh.dropna(subset=["month"])
              .groupby(["provider_name","month"], as_index=False)
              .agg(beds=(bed_col,"max")))

def canonical_fac(nm):
    nm = str(nm).upper()
    if re.search(r"SAGE\s*PARK", nm): return "SAGE PARK"
    if re.search(r"TAYLOR\s*SPRINGS", nm): return "TAYLOR SPRINGS"
    return nm
nh_fac_m["facility"] = nh_fac_m["provider_name"].apply(canonical_fac)

beds_m = (nh_fac_m.pivot_table(index="month", columns="facility", values="beds", aggfunc="max")
                 .rename(columns={"SAGE PARK":"beds_sage_park", "TAYLOR SPRINGS":"beds_taylor_springs"})
                 .sort_index())
if not beds_m.empty: beds_m = beds_m.ffill()
beds_m = beds_m.reset_index()

# ============================ Runs ===========================
runs["month"] = monthify(runs["incident_date"])
runs = runs.dropna(subset=["month"]).copy()

runs["ems"] = pd.to_numeric(runs.get("ems", 0), errors="coerce").fillna(0).astype(int)
runs["is_ems"]  = runs["ems"]
runs["is_fire"] = (1 - runs["is_ems"]).astype(int)

runs["address_norm"] = normalize_address(runs["address"])
runs["facility"] = runs["address_norm"].apply(tag_facility)

# Apartments
runs["property_use_code"] = pd.to_numeric(runs.get("property_use_code", np.nan), errors="coerce")
runs["is_apartment"] = runs["property_use_code"].eq(APT_CODE).astype(int)

# Property-use categories → normalize and bucket
pucat = runs.get("property_use_category")
pucat = pucat.astype(str).str.upper().str.strip() if pucat is not None else pd.Series("", index=runs.index)
runs["pucat_norm"] = pucat

def bucket_use(cat):
    if isinstance(cat, str):
        if cat.startswith(RESIDENTIAL_PREFIX): return "Residential"
        if cat in COMMERCIAL_CATS:             return "Commercial"
        if cat in INDUSTRIAL_CATS:             return "Industrial"
    return "Other"

runs["use_bucket"] = runs["pucat_norm"].apply(bucket_use)

# Facility monthly runs
fac_runs = (runs.dropna(subset=["facility"])
                 .groupby(["month","facility"], as_index=False)
                 .agg(
                     runs_fac_total=("incident_number","count"),
                     runs_fac_ems=("is_ems","sum"),
                     runs_fac_fire=("is_fire","sum"),
                 ))
fac_wide = (fac_runs.pivot_table(index="month", columns="facility",
                                 values=["runs_fac_total","runs_fac_ems","runs_fac_fire"],
                                 aggfunc="sum").sort_index())
if not fac_wide.empty:
    fac_wide.columns = [f"{a}_{b}".lower().replace(" ","_")
                        for (a,b) in fac_wide.columns.to_flat_index()]
    fac_wide = fac_wide.reset_index()
for base in ["runs_fac_total","runs_fac_ems","runs_fac_fire"]:
    for fac in ["sage_park","taylor_springs"]:
        col = f"{base}_{fac}"
        if col not in fac_wide.columns:
            fac_wide[col] = 0

# Monthly totals
tot_runs = (runs.groupby("month", as_index=False)
                 .agg(total_calls=("incident_number","count"),
                      ems_calls=("is_ems","sum"),
                      fire_calls=("is_fire","sum")))

# Apartments monthly (vectorized)
runs["apt_ems"]  = runs["is_apartment"] * runs["is_ems"]
runs["apt_fire"] = runs["is_apartment"] * runs["is_fire"]
apt_runs = (runs.groupby("month", as_index=False)
                .agg(
                    runs_apartment_total=("is_apartment","sum"),
                    runs_apartment_ems=("apt_ems","sum"),
                    runs_apartment_fire=("apt_fire","sum"),
                ))

# Commercial / Industrial / Residential splits from runs
bucket_runs = (runs.groupby(["month","use_bucket"], as_index=False)
                    .agg(
                        ems_calls_bucket=("is_ems","sum"),
                        fire_calls_bucket=("is_fire","sum")
                    ))
bkt_pvt = (bucket_runs.pivot(index="month", columns="use_bucket",
                             values=["ems_calls_bucket","fire_calls_bucket"]).sort_index())
if not bkt_pvt.empty:
    bkt_pvt.columns = [f"{a}_{b.lower()}" for (a,b) in bkt_pvt.columns.to_flat_index()]
    bkt_pvt = bkt_pvt.reset_index()

# Ensure the bucket columns exist (fill 0 if missing)
for c in [
    "ems_calls_bucket_commercial","ems_calls_bucket_industrial","ems_calls_bucket_residential",
    "fire_calls_bucket_commercial","fire_calls_bucket_industrial","fire_calls_bucket_residential"
]:
    if c not in bkt_pvt.columns:
        bkt_pvt[c] = 0

# ============================ Parcels (robust, imputed, ffilled, monthly) =========================
def build_parcel_monthly(parcels_raw: pd.DataFrame) -> pd.DataFrame:
    # --- Normalize time ---
    date_col = "snapshot_month" if "snapshot_month" in parcels_raw.columns else _first_existing(parcels_raw, ["report_month","month","date"])
    parcels = parcels_raw.copy()
    parcels["month"] = monthify(parcels[date_col])

    # --- Identify parcel key (for forward-fill by parcel) ---
    parcel_id_col = _first_existing(parcels, ["parcelno","parcel_id","parcel_number","parid","PARCEL","parcel"])
    if parcel_id_col is None:
        addr_col = _first_existing(parcels, ["situs","situs_full","situsaddr","address","site_addr"])
        parcels["_synthetic_key"] = parcels.get(addr_col, "").astype(str).str.upper().str.strip() + "|" + parcels.get("pclass","").astype(str)
        parcel_id_col = "_synthetic_key"

    # --- Numeric casting ---
    num_cols = ["apprlnd","apprbld","apprtot","area_a","acrea","land_sqft","landuse"]
    parcels = enforce_numeric(parcels, num_cols)
    parcels["pclass"] = parcels["pclass"].astype(str).str.upper()

    # --- Land sqft convenience ---
    if "land_sqft" in parcels.columns:
        parcels["land_sqft_use"] = parcels["land_sqft"]
    elif "acrea" in parcels.columns:
        parcels["land_sqft_use"] = parcels["acrea"] * 43560
    else:
        parcels["land_sqft_use"] = np.nan

    # --- Category flags ---
    lu = pd.to_numeric(parcels["landuse"], errors="coerce")
    is_res = parcels["pclass"].eq("R")
    is_com = parcels["pclass"].eq("C")
    is_ind = parcels["pclass"].eq("I")

    is_sf = is_res & lu.between(SF_BAND[0], SF_BAND[1], inclusive="both")
    is_mf = is_res & lu.between(MF_BAND[0], MF_BAND[1], inclusive="both")

    # --- Base area to use, to be imputed where missing/zero ---
    parcels["area_a_used"] = parcels["area_a"]

    # Helper: category medians for sqft/$
    def median_sqft_per_dollar(mask):
        pool = parcels[mask & (parcels["area_a"].fillna(0) > 0) & (parcels["apprbld"].fillna(0) > 0)].copy()
        if len(pool) < 20:
            return np.nan
        return (pool["area_a"] / pool["apprbld"]).median()

    med_sf  = median_sqft_per_dollar(is_sf)
    med_mf  = median_sqft_per_dollar(is_mf)
    med_com = median_sqft_per_dollar(is_com)
    med_ind = median_sqft_per_dollar(is_ind)

    def fallback_rate(primary, *alts):
        for v in (primary,)+alts:
            if pd.notna(v) and v > 0:
                return float(v)
        return np.nan

    rate_sf  = fallback_rate(med_sf, med_mf, med_com)
    rate_mf  = fallback_rate(med_mf, med_sf, med_com)
    rate_com = fallback_rate(med_com, med_sf, med_mf)
    rate_ind = fallback_rate(med_ind, med_com, med_sf)

    # Coverage ratios (final fallback)
    COV_SF, COV_MF, COV_COM, COV_IND = 0.25, 0.35, 0.30, 0.40

    def impute_area(mask_cat, rate, cov):
        miss = mask_cat & (parcels["area_a_used"].fillna(0) <= 0)
        # 1) sqft/$ * apprbld
        if pd.notna(rate):
            use_val = miss & (parcels["apprbld"].fillna(0) > 0)
            parcels.loc[use_val, "area_a_used"] = parcels.loc[use_val, "apprbld"] * rate
        # 2) land_sqft * coverage ratio
        use_land = miss & parcels["land_sqft_use"].notna() & (parcels["land_sqft_use"] > 0)
        parcels.loc[use_land, "area_a_used"] = parcels.loc[use_land, "land_sqft_use"] * cov
        # 3) category median area (as last resort)
        cat_med = parcels.loc[mask_cat & (parcels["area_a"].fillna(0) > 0), "area_a"].median()
        use_med = miss & parcels["area_a_used"].isna()
        if pd.notna(cat_med) and cat_med > 0:
            parcels.loc[use_med, "area_a_used"] = cat_med

    impute_area(is_sf,  rate_sf,  COV_SF)
    impute_area(is_mf,  rate_mf,  COV_MF)
    impute_area(is_com, rate_com, COV_COM)
    impute_area(is_ind, rate_ind, COV_IND)

    # --- Remove MF-like records from Commercial only (avoid double count) ---
    mf_cues = ["APART","APT","MULTI","DUPLEX","TRIPLEX","TOWNHOME","CONDO"]
    text_cols = [c for c in ["descr1","descr2","descr3","proptyp"] if c in parcels.columns]
    mf_like = np.zeros(len(parcels), dtype=bool)
    for c in text_cols:
        s = parcels[c].astype(str).str.upper().fillna("")
        for cue in mf_cues:
            mf_like |= s.str.contains(cue, na=False)
    suspect_mf_in_com = is_com & (mf_like | lu.between(MF_BAND[0], MF_BAND[1], inclusive="both"))

    # --- Keep minimal columns for ffill ---
    parcel_id_col = _first_existing(parcels, ["parcelno","parcel_id","parcel_number","parid","PARCEL","parcel"]) or "_synthetic_key"
    p = parcels[[parcel_id_col, "month", "area_a_used", "pclass", "landuse"]].copy()

    # Assign category areas (fresh from area_a_used)
    p["res_sf_area_sqft"] = np.where(is_sf, parcels["area_a_used"], 0.0)
    p["res_mf_area_sqft"] = np.where(is_mf, parcels["area_a_used"], 0.0)
    p["com_area_sqft"]    = np.where(is_com & (~suspect_mf_in_com), parcels["area_a_used"], 0.0)
    p["ind_area_sqft"]    = np.where(is_ind, parcels["area_a_used"], 0.0)

    # Forward-fill per parcel across months
    p = p.sort_values([parcel_id_col, "month"])
    p[["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]] = (
        p.groupby(parcel_id_col)[["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]]
         .ffill()
    )

    # Monthly aggregates after ffill
    parcel_m = (p.groupby("month", as_index=False)
                  .agg(res_sf_area_sqft=("res_sf_area_sqft","sum"),
                       res_mf_area_sqft=("res_mf_area_sqft","sum"),
                       com_area_sqft=("com_area_sqft","sum"),
                       ind_area_sqft=("ind_area_sqft","sum"))
                .sort_values("month"))

    # Proxies & k-sqft
    for c in ["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]:
        parcel_m[f"{c}_k"] = parcel_m[c] / 1000.0

    parcel_m["total_res_area_sqft_k"] = parcel_m["res_sf_area_sqft_k"] + parcel_m["res_mf_area_sqft_k"]
    parcel_m["est_residents_sf"]     = parcel_m["res_sf_area_sqft_k"] * DENSITY_RES_PEOPLE_PER_1K_SQFT
    parcel_m["est_residents_mf"]     = parcel_m["res_mf_area_sqft_k"] * DENSITY_MF_PEOPLE_PER_1K_SQFT
    parcel_m["est_residents_total"]  = parcel_m["est_residents_sf"] + parcel_m["est_residents_mf"]
    parcel_m["est_workers_com"]      = parcel_m["com_area_sqft_k"]   * DENSITY_COM_JOBS_PER_1K_SQFT
    parcel_m["est_workers_ind"]      = parcel_m["ind_area_sqft_k"]   * DENSITY_IND_JOBS_PER_1K_SQFT
    parcel_m["est_workers_total"]    = parcel_m["est_workers_com"] + parcel_m["est_workers_ind"]

    # Lagging (6 months) for predictors used in R
    for c in [
        "res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k","total_res_area_sqft_k",
        "est_residents_mf","est_residents_total",
        "est_workers_com","est_workers_ind","est_workers_total"
    ]:
        parcel_m[f"{c}_lag6"] = parcel_m[c].shift(6)

    return parcel_m

parcel_m = build_parcel_monthly(parcels_raw)

# ============================ Compose panel ============================
panel = (tot_runs
         .merge(apt_runs, on="month", how="left")
         .merge(fac_wide, on="month", how="left")
         .merge(beds_m, on="month", how="left")
         .merge(bkt_pvt, on="month", how="left")
         .merge(parcel_m, on="month", how="left")
         .sort_values("month").reset_index(drop=True))

# Build continuous monthly index (min→max across all sources), ffill beds
if not panel.empty:
    full_idx = pd.date_range(panel["month"].min(), panel["month"].max(), freq="MS")
    panel = (panel.set_index("month").reindex(full_idx).rename_axis("month").reset_index())

for c in ["beds_sage_park","beds_taylor_springs"]:
    if c in panel.columns: panel[c] = panel[c].ffill()

# Fill NA zeros where appropriate
for c in [
    "runs_apartment_total","runs_apartment_ems","runs_apartment_fire",
    "runs_fac_total_sage_park","runs_fac_ems_sage_park","runs_fac_fire_sage_park",
    "runs_fac_total_taylor_springs","runs_fac_ems_taylor_springs","runs_fac_fire_taylor_springs",
    "ems_calls_bucket_commercial","ems_calls_bucket_industrial","ems_calls_bucket_residential",
    "fire_calls_bucket_commercial","fire_calls_bucket_industrial","fire_calls_bucket_residential",
    "ems_calls","fire_calls","total_calls"
]:
    if c in panel.columns:
        panel[c] = panel[c].fillna(0).astype(int)

# Convenience labels (facility totals)
panel["runs_sage_total"]   = panel.get("runs_fac_total_sage_park", 0)
panel["runs_taylor_total"] = panel.get("runs_fac_total_taylor_springs", 0)
panel["runs_sage_ems"]     = panel.get("runs_fac_ems_sage_park", 0)
panel["runs_taylor_ems"]   = panel.get("runs_fac_ems_taylor_springs", 0)
panel["runs_sage_fire"]    = panel.get("runs_fac_fire_sage_park", 0)
panel["runs_taylor_fire"]  = panel.get("runs_fac_fire_taylor_springs", 0)

# Residential (non‑apt, non‑NH) outcomes (can be negative if rare misclass — clamp at 0)
panel["ems_residential_other"] = (
    panel["ems_calls"]
    - panel.get("runs_apartment_ems",0)
    - panel.get("runs_sage_ems",0)
    - panel.get("runs_taylor_ems",0)
).clip(lower=0)
panel["fire_residential_other"] = (
    panel["fire_calls"]
    - panel.get("runs_apartment_fire",0)
    - panel.get("runs_sage_fire",0)
    - panel.get("runs_taylor_fire",0)
).clip(lower=0)

# Save
panel.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}  (rows={len(panel)}, cols={panel.shape[1]})")

# Quick sanity preview
checks = [
    "res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft",
    "res_sf_area_sqft_k","res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k",
    "res_mf_area_sqft_k_lag6","com_area_sqft_k_lag6","ind_area_sqft_k_lag6","total_res_area_sqft_k_lag6",
    "est_residents_mf","est_residents_mf_lag6","est_residents_total","est_residents_total_lag6",
    "est_workers_com","est_workers_com_lag6","est_workers_ind","est_workers_ind_lag6",
    "ems_calls_bucket_commercial","fire_calls_bucket_commercial",
    "ems_calls_bucket_industrial","fire_calls_bucket_industrial",
    "runs_sage_total","runs_taylor_total","runs_apartment_ems","runs_apartment_fire",
    "ems_residential_other","fire_residential_other"
]
print(panel[["month"] + [c for c in checks if c in panel.columns]].head(8))

Saved: C:\Repositories\jefferson-township-run-forecasting\data\clean\panel_monthly_with_parcels.csv  (rows=84, cols=55)
       month  res_sf_area_sqft  res_mf_area_sqft  com_area_sqft  \
0 2018-08-01      1.153223e+07          734847.0   8.045859e+06   
1 2018-09-01      1.153223e+07          734847.0   8.045859e+06   
2 2018-10-01      1.153098e+07          734847.0   8.045859e+06   
3 2018-11-01      1.018856e+07          734847.0   5.640114e+06   
4 2018-12-01      1.051754e+07          734847.0   5.636349e+06   
5 2019-01-01      1.051598e+07          734847.0   5.636349e+06   
6 2019-02-01      1.172388e+07          734847.0   8.042094e+06   
7 2019-03-01      1.175154e+07          734847.0   8.042094e+06   

   ind_area_sqft  res_sf_area_sqft_k  res_mf_area_sqft_k  com_area_sqft_k  \
0   3.746361e+06        11532.234846             734.847      8045.858990   
1   3.746361e+06        11532.234846             734.847      8045.858990   
2   3.746438e+06        11530.977846         